In [1]:
%pip install qiskit==1.2.4
%pip install qiskit-aer==0.15.1
%pip install pylatexenc==2.10

from qiskit import QuantumCircuit
from qiskit.converters import circuit_to_gate
from qiskit.visualization import array_to_latex
from qiskit.quantum_info import Operator
from qiskit.quantum_info import Statevector
from qiskit import transpile
from qiskit.providers.basic_provider import BasicSimulator
from qiskit.visualization import plot_histogram
from qiskit.circuit import ControlledGate

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.8/4.8 MB 32.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.2/2.2 MB 22.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.5/54.5 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.6/49.6 MB 11.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 85.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 162.6/162.6 kB 12.1 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for pylatexenc: filename=pylatexenc-2.10-py3-none-any.whl size=136817 sha256=42fba6849289bdbac9fb4783e0ed53e38b98e3abb4859e642a08a3c4a385ba7c
  Stored in directory: /root/.cache/pip/wheels/06/3e/78/fa1588c1ae991bbfd814af2bcac6cef7a178beee1939180d46
Successfully built pylatexenc


In [2]:
# The aim of the assignment is to simulate the BB84 key distribution protocol.
# This notebook is for a simulation of the protocol without an attacker.

# Generates a list of true random bits.
def get_quantum_random_bits(num_bits):
  qc = QuantumCircuit(num_bits, num_bits)
  # Apply Hadamard gate to all qubits to put them in |+> state
  qc.h(range(num_bits))
  # Measure all qubits
  qc.measure(range(num_bits), range(num_bits))

  # Run the circuit
  backend = BasicSimulator()
  job = backend.run(transpile(qc, backend), shots=1, memory=True)

  # Qiskit outputs strings right-to-left. It is reversed [::-1] to match
  # left-to-right indexing for easier tracking.
  bit_string = job.result().get_memory()[0][::-1]
  return [int(b) for b in bit_string]


In [3]:
# Formats and prints the results in a clear table for easy understanding.
def print_bb84_table(alice_bits, alice_bases, bob_bases, bob_results):
    basis_sym = {0: '+ (Std)', 1: 'x (Diag)'}
    print("=" * 75)
    print(f"{'Qubit':<7} | {'Alice Bit':<10} | {'Alice Basis':<12} | {'Bob Basis':<12} | {'Bob Bit':<8} | {'Key Action'}")
    print("-" * 75)

    for i in range(len(alice_bits)):
        # If Alice and Bob randomly chose the same basis, they keep the bit.
        if alice_bases[i] == bob_bases[i]:
            action = f"Keep -> {alice_bits[i]}"
        else:
            action = "Discard"

        print(f"{i:<7} | {alice_bits[i]:<10} | {basis_sym[alice_bases[i]]:<12} | "
              f"{basis_sym[bob_bases[i]]:<12} | {bob_results[i]:<8} | {action}")
    print("=" * 75)

In [4]:
def sim_bb84_plain(num_bits=16):
    print(f"--- Starting BB84 Simulation with {num_bits} qubits ---\n")

    # Alice generates her random raw bits and her random encoding bases.
    # Basis 0 = Standard Basis (+): States |0> or |1>
    # Basis 1 = Diagonal Basis (x): States |+> or |->
    alice_bits = get_quantum_random_bits(num_bits)
    alice_bases = get_quantum_random_bits(num_bits)

    # Alice prepares her quantum circuit
    qc = QuantumCircuit(num_bits, num_bits)

    for i in range(num_bits):
        # 1. Encode Bit: Qubits start at |0>. If Alice wants to send a '1',
        # she applies an X gate (Quantum NOT) to flip it to |1>.
        if alice_bits[i] == 1:
            qc.x(i)

        # 2. Encode Basis: If Alice chose the Diagonal basis (1),
        # she applies a Hadamard (H) gate.
        # This turns |0> into |+> and |1> into |->.
        if alice_bases[i] == 1:
            qc.h(i)

    # Bob has no idea what bases Alice used, so he randomly guesses.
    bob_bases = get_quantum_random_bits(num_bits)

    for i in range(num_bits):
        # If Bob decides to measure in the Diagonal basis (1), he must apply
        # an H gate before standard measurement to rotate the state back.
        if bob_bases[i] == 1:
            qc.h(i)
        # Bob measures the qubit, collapsing its quantum state.
        qc.measure(i, i)

    # Execute the quantum circuit
    backend = BasicSimulator()
    job = backend.run(transpile(qc, backend), shots=1, memory=True)

    # Read Bob's results
    bob_results = [int(bit) for bit in job.result().get_memory()[0][::-1]]

    # Alice and Bob publicly announce the BASES they used (but NOT the bits!).
    # They discard any bits where their bases didn't match.
    shared_key_alice = []
    shared_key_bob = []

    for i in range(num_bits):
        if alice_bases[i] == bob_bases[i]:
            shared_key_alice.append(alice_bits[i])
            shared_key_bob.append(bob_results[i])

    # Display the visual tables
    print_bb84_table(alice_bits, alice_bases, bob_bases, bob_results)

    # Verification
    print(f"\nTotal bits matched in basis: {len(shared_key_alice)} out of {num_bits}.")
    print(f"Alice's Key: {shared_key_alice}")
    print(f"Bob's Key:   {shared_key_bob}")
    print(f"Keys Match Perfectly? {'YES' if shared_key_alice == shared_key_bob else 'NO'}")

    return shared_key_alice

In [5]:
final_key = sim_bb84_plain(20)

--- Starting BB84 Simulation with 20 qubits ---

Qubit   | Alice Bit  | Alice Basis  | Bob Basis    | Bob Bit  | Key Action
---------------------------------------------------------------------------
0       | 1          | x (Diag)     | + (Std)      | 1        | Discard
1       | 0          | + (Std)      | + (Std)      | 0        | Keep -> 0
2       | 0          | + (Std)      | x (Diag)     | 1        | Discard
3       | 0          | x (Diag)     | x (Diag)     | 0        | Keep -> 0
4       | 0          | x (Diag)     | + (Std)      | 1        | Discard
5       | 1          | + (Std)      | x (Diag)     | 0        | Discard
6       | 0          | x (Diag)     | x (Diag)     | 0        | Keep -> 0
7       | 0          | x (Diag)     | x (Diag)     | 0        | Keep -> 0
8       | 0          | x (Diag)     | x (Diag)     | 0        | Keep -> 0
9       | 1          | x (Diag)     | x (Diag)     | 1        | Keep -> 1
10      | 1          | + (Std)      | + (Std)      | 1        | Keep

## Output Explanation
This first simulation shows how Alice and Bob establish a shared secret key when the channel is untouched.

**Alice Bit & Basis:** Alice generates a random bit and encodes it into a quantum state using a randomly chosen basis: either standard `+` or diagonal `x`.

**Bob Basis:** Bob receives the qubits but doesn't know Alice's bases. He randomly guesses which basis to use for his measurements.

**Bob Bit:** The result of Bob's measurement.

- If Bob guesses correctly: His measurement perfectly matches Alice's encoded bit (deterministic).

- If Bob guesses incorrectly: He forces the qubit into a different basis, resulting in a completely random `0` or `1` measurement (probabilistic).

**Sifting:** Alice and Bob publicly share only the bases they used, not the bits. If their bases match, they Keep the bit for their final key. If they differ, they Discard it.

**The Result:**
Out of 20 qubits sent, Alice and Bob randomly chose the exact same basis 11 times. After discarding the 9 mismatched qubits, their resulting 11-bit keys match perfectly. The channel is secure, and they now share a secret cryptographic key.